# Image Quality Assessment -- Training Notebook

Official Colab training notebook for pipeline Step 1 (Image Quality Assessment). This notebook
only orchestrates -- every model, dataset, training, and evaluation step calls directly into
existing project modules (`image_quality_dataset.py`, `image_quality_model.py`,
`train_image_quality.py`, `evaluate_image_quality.py`, `image_quality_inference.py`,
`training/`, `evaluation/`) and the new `colab/` workflow modules
(`setup.py`, `colab_config.py`, `verify_environment.py`, `verify_dataset.py`,
`experiment_manager.py`). See **`colab/README.md`** for full documentation: folder structure,
the Google Drive layout, how to resume a run, how to launch TensorBoard, and troubleshooting.

**Model:** EfficientNetB0, 3-class softmax (`Good` / `Usable` / `Reject`)
**Dataset:** EyeQ, read from Google Drive (`datasets/EyeQ/raw`, never modified)
**Outputs:** written entirely to Google Drive, isolated per timestamped experiment under
`experiments/IQA/` -- nothing important is left on the ephemeral Colab VM.

**Before running:** `Runtime > Change runtime type > Hardware accelerator > GPU`, and confirm
your Drive contains `MyDrive/DiabeticRetinopathy/datasets/EyeQ/raw/{train,test}` (see
`colab/README.md` for the full verified layout).

### Bootstrap

Every cell below calls into `colab/` modules (`setup.py`, `colab_config.py`, ...), which only
become importable once the repository is cloned. This cell performs the minimal clone +
`sys.path` setup needed for that; Section 1 (`setup.setup()`) repeats the clone as an idempotent
`git pull` once it's available, so the two never drift out of sync (see `colab/setup.py`'s
module docstring for why this duplication is intentional and minimal).

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

## 1. Setup

`setup.setup()` mounts Google Drive, clones/updates the repository, installs
`requirements.txt`, enters the repository, and points `EYEQ_RAW_DIR` at Drive -- see
`colab/setup.py`. Nothing here is reimplemented; this is the one call the notebook makes.

In [ ]:
import setup

setup_info = setup.setup()

## 2. Verification

`verify_environment.verify_all()` checks Python/TensorFlow versions, the repository path,
Google Drive, required packages, GPU availability, CUDA, and mixed precision -- aborting with a
clear error on the first failure (see `colab/verify_environment.py`).

In [ ]:
import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)

`verify_dataset.verify_eyeq_dataset()` checks the dataset directory, `labels.csv`, image
directories, image counts, missing images, a corrupted-image spot check, and class
distribution -- aborting with a clear error if the dataset isn't usable (see
`colab/verify_dataset.py`).

In [ ]:
import verify_dataset

dataset_report = verify_dataset.verify_eyeq_dataset(colab_config.EYEQ_RAW_DIR)
print("Train class distribution:", dataset_report.train.class_distribution)
print("Test class distribution:", dataset_report.test.class_distribution)

## 3. Dataset Loading

Hyperparameters, a GPU-memory-aware batch size (32, falling back to 16 below 12 GB), and a new
isolated experiment folder under `experiments/IQA/<timestamp>/` (see
`colab/experiment_manager.py` -- every run gets its own folder and previous runs are never
overwritten). Set `RESUME_EXPERIMENT_DIR` to an existing experiment's path to resume that run
instead of starting a new one. Dataset loading itself calls
`image_quality_dataset.load_eyeq_datasets()` unmodified.

In [ ]:
import environment

IMAGE_SIZE = (224, 224)
EPOCHS = 50
LEARNING_RATE = 1e-4
FREEZE_LAYERS = 100
RANDOM_SEED = 42  # matches image_quality_dataset.load_eyeq_datasets's own default split seed
RESUME_EXPERIMENT_DIR = None  # or an existing experiments/IQA/<timestamp> path to resume it


def select_batch_size(preferred=32, fallback=16, min_total_mib=12000):
    total_mib = environment.get_gpu_memory_mib()
    if total_mib is None:
        print(f"No GPU / could not query GPU memory -- using batch size {fallback}.")
        return fallback
    print(f"Detected GPU memory: {total_mib} MiB")
    if total_mib >= min_total_mib:
        print(f"Using batch size {preferred}.")
        return preferred
    print(f"GPU memory below {min_total_mib} MiB threshold -- falling back to batch size {fallback}.")
    return fallback


BATCH_SIZE = select_batch_size()

In [ ]:
import experiment_manager

experiment = experiment_manager.resolve_experiment(
    colab_config.IQA_EXPERIMENTS_DIR,
    colab_config.REPO_DIR,
    resume_from=RESUME_EXPERIMENT_DIR,
    dataset_path=colab_config.EYEQ_RAW_DIR,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    image_size=list(IMAGE_SIZE),
    freeze_layers=FREEZE_LAYERS,
    random_seed=RANDOM_SEED,
)
print(f"Experiment root: {experiment.root}")

In [ ]:
from image_quality_dataset import load_eyeq_datasets

train_ds, val_ds, class_weights = load_eyeq_datasets(
    raw_dir=colab_config.EYEQ_RAW_DIR, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
)
print(f"Class weights (train split): {class_weights}")

## 4. Model Creation

Builds the model once here to inspect the architecture before committing to a full run.
`train_image_quality.train()` (Section 5) builds and compiles its own instance from the same
`image_quality_model.build_iqa_model` factory -- this preview model is discarded, not reused.

In [ ]:
from image_quality_model import build_iqa_model

preview_model = build_iqa_model(
    input_shape=(*IMAGE_SIZE, 3), learning_rate=LEARNING_RATE, freeze_layers=FREEZE_LAYERS,
)
preview_model.summary()
del preview_model

## 5. Training

Calls `train_image_quality.train()` directly, pointing it at this experiment's Drive folders --
dataset loading, model construction, mixed precision, checkpointing, early stopping,
`ReduceLROnPlateau`, TensorBoard logging, and resume support are all handled inside that
function and `training.Trainer`; nothing here reimplements any of it. `checkpoints/` and
`logs/` under the experiment root are populated directly by `training.Trainer`;
`archive_tensorboard_logs()` then copies the TensorBoard event files into `tensorboard/` (see
`colab/experiment_manager.py`'s module docstring for why these are two separate folders).

In [ ]:
from train_image_quality import train

model, history, exported_path = train(
    raw_dir=colab_config.EYEQ_RAW_DIR,
    run_dir=experiment.root,
    export_path=os.path.join(colab_config.IQA_EXPORTED_MODELS_DIR, "best_model.keras"),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    freeze_layers=FREEZE_LAYERS,
    resume=RESUME_EXPERIMENT_DIR is not None,
)

experiment_manager.archive_tensorboard_logs(experiment)
print(f"\nBest model exported to: {exported_path}")

### TensorBoard

Points at this experiment's live log directory.

In [ ]:
logs_dir = experiment.logs_dir
%load_ext tensorboard
%tensorboard --logdir $logs_dir

## 6. Evaluation

Training curves, then `evaluate_image_quality.evaluate()` over the held-out
`datasets/EyeQ/raw/test/` split (never used for training or validation above), writing
confusion matrix / ROC / calibration plots and the full report into this experiment's
`evaluation/` folder. A small sample-predictions grid (via `image_quality_inference.predict_quality`)
is saved to `predictions/`.

In [ ]:
import matplotlib.pyplot as plt

metrics = [k for k in history.history if not k.startswith("val_")]
fig, axes = plt.subplots(len(metrics), 1, figsize=(8, 4 * len(metrics)))
if len(metrics) == 1:
    axes = [axes]
for ax, metric in zip(axes, metrics):
    ax.plot(history.history[metric], label=f"train_{metric}")
    val_key = f"val_{metric}"
    if val_key in history.history:
        ax.plot(history.history[val_key], label=val_key)
    ax.set_title(metric)
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(experiment.evaluation_dir, "training_history.png"), bbox_inches="tight")
plt.show()

In [ ]:
from evaluate_image_quality import evaluate

test_report = evaluate(
    raw_dir=colab_config.EYEQ_RAW_DIR,
    model_path=exported_path,
    batch_size=BATCH_SIZE,
    output_dir=experiment.evaluation_dir,
)

In [ ]:
from IPython.display import Image, display

display(Image(filename=os.path.join(experiment.evaluation_dir, "confusion_matrix.png")))
display(Image(filename=os.path.join(experiment.evaluation_dir, "roc_curves.png")))
display(Image(filename=os.path.join(experiment.evaluation_dir, "calibration_curve.png")))

In [ ]:
from image_quality_inference import load_iqa_model, predict_quality
import image_quality_dataset as iqd

inference_model = load_iqa_model(exported_path)
test_df = iqd._read_labels(colab_config.EYEQ_RAW_DIR, "test")
sample_rows = test_df.sample(n=min(8, len(test_df)), random_state=7)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    result = predict_quality(row["path"], model=inference_model, image_size=IMAGE_SIZE)
    image = iqd._decode_image(row["path"], IMAGE_SIZE).numpy().astype("uint8")
    true_label = iqd.QUALITY_CLASSES[int(row["quality"])]
    outcome = "MATCH" if result["label"] == true_label else "MISMATCH"
    ax.imshow(image)
    ax.set_title(f"{outcome}\ntrue={true_label} pred={result['label']}\nconf={result['confidence']:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
predictions_path = os.path.join(experiment.predictions_dir, "sample_predictions.png")
plt.savefig(predictions_path, bbox_inches="tight")
plt.show()
print(f"Saved sample predictions to {predictions_path}")

## 7. Export

The best checkpoint was already exported to `exported_path` -- inside
`exported_models/IQA/` on Google Drive -- by `train_image_quality.train()` in Section 5; this
section only confirms it and summarizes every artifact this run produced.

In [ ]:
assert os.path.exists(exported_path), "Export path missing -- did training complete?"

print("=" * 72)
print("IMAGE QUALITY ASSESSMENT -- RUN SUMMARY")
print("=" * 72)
print(f"Exported model:   {exported_path}  ({os.path.getsize(exported_path) / 1e6:.2f} MB)")
print(f"Experiment root:  {experiment.root}")
print(f"  checkpoints/:   {experiment.checkpoints_dir}")
print(f"  logs/ (TB):     {experiment.logs_dir}")
print(f"  tensorboard/:   {experiment.tensorboard_dir}")
print(f"  evaluation/:    {experiment.evaluation_dir}")
print(f"  predictions/:   {experiment.predictions_dir}")
print(f"  metadata.json:  {experiment.metadata_path}")

print("\nHeld-out test-split metrics:")
print(f"  accuracy: {test_report.accuracy:.4f}  f1: {test_report.f1:.4f}  "
      f"auc: {test_report.auc:.4f}  qwk: {test_report.quadratic_weighted_kappa:.4f}")

print("\nNext step in the pipeline (see PROJECT_CODE.md): Step 2 -- Image Preprocessing,")
print("gated by this model's quality predictions via image_quality_inference.py.")